In [247]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# 1. Read Data and Set Boundaries

In [ ]:
df = pd.read_csv("house_prices.csv")
df.isnull().sum().sort_values(ascending=False).head(20)
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 20)

# 2. Cleaning and Handling Missing Values

In [249]:
df = df.drop(columns=['PoolQC', 'MiscFeature', 'Alley', 'Fence'])
empty_columns = ['MasVnrType', 'FireplaceQu', 'GarageQual', 'GarageFinish', 
                            'GarageType', 'GarageCond', 'BsmtFinType2', 'BsmtExposure', 
                            'BsmtCond', 'BsmtQual', 'BsmtFinType1']
for col in empty_columns:
    df[col] = df[col].fillna('None')
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['GarageYrBlt'].median())
df['MasVnrArea'] = df['MasVnrArea'].fillna(df['MasVnrArea'].median())
df = df.dropna(subset=['Electrical'])
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,Reg,Lvl,AllPub,FR2,...,0,0,0,0,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,0,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,IR1,Lvl,AllPub,Corner,...,272,0,0,0,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,IR1,Lvl,AllPub,FR2,...,0,0,0,0,0,12,2008,WD,Normal,250000


# 3. Train / Split

In [250]:
columns = df.select_dtypes(include = ["object"]).columns
df = pd.get_dummies(df, columns, drop_first = True)

x = df.drop(columns = ["SalePrice"])
y = df["SalePrice"]
y = np.log(y)

x_train, x_test, y_train,y_test = train_test_split(x, y, train_size= 0.8, random_state = 21)

# 4. Modelling (XGBoost with Tuned Parameters)

In [251]:
model = xgb.XGBRegressor( n_estimators=500, learning_rate=0.05, max_depth=4, random_state= 21)
model.fit(x_train, y_train)
model.score(x_test, y_test)

0.8597343453884848

# 5. Modelling (RandomForest)

In [252]:
rf = RandomForestRegressor(random_state=21)
rf_model = rf.fit(x_train, y_train)
rf_model.score(x_test, y_test)

0.8366899780758553

# 6. Prediction vs. Actual Value

In [253]:
sample = x_test.sample()
prediction = model.predict(sample)
real = y_test.loc[sample.index].values

print("-------------XGBoost-------------")
print("Prediction Price:", np.exp(prediction)[0], "$")
print("Real Price:", np.exp(real)[0], "$")

rf_sample = x_test.sample()
rf_prediction = rf_model.predict(rf_sample)
rf_real = y_test.loc[rf_sample.index].values

print("-------------RandomForest-------------")
print("Prediction Price:", np.exp(rf_prediction)[0], "$")
print("Real Price:", np.exp(rf_real)[0], "$")

-------------XGBoost-------------
Prediction Price: 124977.08 $
Real Price: 96999.99999999997 $
-------------RandomForest-------------
Prediction Price: 195362.03372765228 $
Real Price: 204749.99999999997 $


# 7. Mean Absolute Error

In [254]:
predictions = np.exp(model.predict(x_test))
reals = np.exp(y_test)

mae = mean_absolute_error(reals, predictions)

print("--------------------XGBoost--------------------")
print("Average Error (MAE) for XGBoost:", mae, "$")

rf_predictions = np.exp(rf_model.predict(x_test))
rf_reals = np.exp(y_test)

rf_mae = mean_absolute_error(rf_reals, rf_predictions)
print("--------------------RandomForest--------------------")
print("Average Error (MAE) for RandomForest:", rf_mae, "$")

--------------------XGBoost--------------------
Average Error (MAE) for XGBoost: 14819.434383026535 $
--------------------RandomForest--------------------
Average Error (MAE) for RandomForest: 16553.098280586353 $
